# Notebook 02 — Pandas and Air-Quality Data Foundations
## การจัดเตรียม ตรวจสอบ และวิเคราะห์ข้อมูลคุณภาพอากาศของ PCD เบื้องต้น

**ระดับผู้เรียน:** ปริญญาตรีปี 3–4 / ปริญญาโท  
**สาขา:** สิ่งแวดล้อม ภูมิศาสตร์ ภูมิสารสนเทศ อุตุนิยมวิทยา และสาขาที่เกี่ยวข้อง  
**Platform:** Google Colab

---

# แนวคิดของบทเรียน

Notebook 01 ทำให้เราได้ **canonical teaching dataset**

Notebook 02 จะเริ่มจากข้อมูล PCD/Air4Thai ใน Excel:

```text
2021.xlsx
2022.xlsx
2023.xlsx
2024.xlsx
2025.xlsx
```

แล้วค่อย ๆ เปลี่ยนจาก:

```text
Excel files
    ↓
Pandas DataFrame
    ↓
Schema inspection
    ↓
Data types
    ↓
Datetime
    ↓
Station
    ↓
Pollutants
    ↓
Missing / Duplicate / QC
    ↓
Multi-year harmonization
    ↓
Daily / Monthly summaries
    ↓
Basic visualization
    ↓
Analysis-ready teaching tables
```

บทนี้ยังไม่ทำ spatial analysis อย่างจริงจัง

คำถามหลักของบทนี้คือ:

> ข้อมูลคุณภาพอากาศมีโครงสร้างอย่างไร  
> คุณภาพข้อมูลเป็นอย่างไร  
> และเราจะเตรียมข้อมูลหลายปีให้พร้อมสำหรับการวิเคราะห์อย่างมีหลักฐานได้อย่างไร?

# Learning Outcomes

เมื่อจบ Notebook นี้ นิสิตควรสามารถ:

1. ใช้ `pd.read_excel()` อ่านข้อมูล PCD ได้
2. อธิบายความแตกต่างระหว่าง row, column, observation และ variable ได้
3. ตรวจ schema ของ Excel หลายปีได้
4. ตรวจ data type ด้วย `dtypes` และ `info()` ได้
5. แปลงวันเวลาเป็น `datetime`
6. แปลง pollutant fields เป็น numeric อย่างปลอดภัย
7. ตรวจ missing values
8. ตรวจ duplicate records
9. ตรวจ station IDs
10. เชื่อม station metadata ด้วย attribute join
11. harmonize schema หลายปีโดยไม่ทำลาย raw data
12. แปลงข้อมูลจาก wide format เป็น long format
13. สร้าง daily และ monthly summaries
14. ใช้ `groupby()`, `agg()`, `resample()` และ `pivot_table()`
15. สร้าง time-series, histogram และ boxplot
16. สร้าง QC flags แทนการลบค่าที่สงสัยแบบอัตโนมัติ
17. อธิบายความต่างระหว่าง missing, invalid และ extreme values
18. สร้าง analysis-ready outputs สำหรับ Notebook ต่อไป

# หลักการทางวิทยาศาสตร์ที่ต้องจำ

## 1. Missing ≠ Zero

ถ้า PM₂.₅ เป็น `NaN`

ไม่ได้แปลว่า:

```text
PM₂.₅ = 0 µg/m³
```

แต่แปลว่า:

> ไม่มีค่าที่ใช้ได้ใน record นั้น

---

## 2. High concentration ≠ Error

ค่ามลพิษที่สูงมากอาจเป็น:

- pollution episode จริง
- biomass burning
- stagnant atmosphere
- regional transport
- urban/industrial emission
- sensor/data problem

ดังนั้น:

> **ห้ามลบค่าที่สูงเพียงเพราะดูเหมือน outlier**

เราจะสร้าง QC flags และตรวจบริบทก่อน

---

## 3. Station observation ≠ Province mean

สถานีเป็น **point measurement**

ค่าของสถานีหนึ่งแห่งไม่ใช่ค่าเฉลี่ยพื้นที่ทั้งจังหวัดโดยอัตโนมัติ

---

## 4. Data availability affects statistics

ถ้าสถานีหนึ่งมีข้อมูล 360 วัน แต่อีกสถานีมีเพียง 40 วัน  
ค่าเฉลี่ยทั้งสองสถานีไม่ได้มีความน่าเชื่อถือในระดับเดียวกัน

ดังนั้นเราต้องรายงาน:

```text
n observations
data completeness
date coverage
```

ควบคู่กับค่าเฉลี่ยเสมอ

In [ ]:
# CELL 1 — Install packages used in this notebook

!pip -q install -U \
    openpyxl \
    pyarrow

In [ ]:
# CELL 2 — Mount Google Drive

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

In [ ]:
# CELL 3 — Imports

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)

print("pandas:", pd.__version__)

# 2.1 Course Paths

Notebook นี้ **ไม่ดาวน์โหลดข้อมูลจากเว็บไซต์ภายนอก**

ใช้ข้อมูลที่ Notebook 01 เตรียมไว้ที่:

```text
MyDrive/
└── Teaching_PCD_Environmental_GIS/
    └── 01_course_data/
        └── v1/
            └── dataset/
```

หาก folder นี้ไม่มี แสดงว่ายังไม่ได้รัน Notebook 01 สำเร็จ

In [ ]:
# CELL 4 — Paths

DATASET_VERSION = "v1"

BASE_DIR = Path(
    "/content/drive/MyDrive/"
    "Teaching_PCD_Environmental_GIS"
)

DATASET_DIR = (
    BASE_DIR
    / "01_course_data"
    / DATASET_VERSION
    / "dataset"
)

OUTPUT_DIR = (
    BASE_DIR
    / "02_output"
    / "Notebook_02"
)

FIGURE_DIR = (
    BASE_DIR
    / "03_figures"
    / "Notebook_02"
)

for folder in [
    OUTPUT_DIR,
    FIGURE_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

if not DATASET_DIR.exists():

    raise FileNotFoundError(
        "Canonical dataset folder was not found.\n"
        "Run Notebook 01 first:\n"
        "01_get_course_dataset_from_GitHub.ipynb"
    )

print("Dataset folder:")
print(DATASET_DIR)

print("\nOutput folder:")
print(OUTPUT_DIR)

# 2.2 Locate Canonical Files

เราไม่ hard-code path ลึกเกินไป

ใช้ชื่อไฟล์เพื่อค้นหา canonical files ที่ Notebook 01 extract ไว้

In [ ]:
# CELL 5 — File locator

def find_unique_file(
    root,
    filename,
    required=True,
):
    matches = list(
        Path(
            root
        ).rglob(
            filename
        )
    )

    if len(
        matches
    ) == 1:

        return matches[
            0
        ]

    if len(
        matches
    ) == 0:

        if required:

            raise FileNotFoundError(
                f"Required file not found: {filename}"
            )

        return None

    raise RuntimeError(
        f"Multiple copies found for {filename}: "
        + str(
            matches
        )
    )


PCD_STATION_CSV = find_unique_file(
    DATASET_DIR,
    "pcd_air4thai_stations.csv",
)

EXCEL_FILES = {
    year:
        find_unique_file(
            DATASET_DIR,
            f"{year}.xlsx",
        )
    for year
    in range(
        2021,
        2026
    )
}


print("PCD station metadata:")
print(PCD_STATION_CSV)

print("\nPCD annual Excel:")
for year, path in EXCEL_FILES.items():
    print(year, "->", path)

# 2.3 Station Metadata Before Pollution Data

ก่อนอ่านค่ามลพิษ เราควรรู้ว่า station table มีอะไร

Canonical station metadata มีประโยชน์สำหรับ:

- ตรวจ station ID
- ตรวจ latitude/longitude
- ดู station name
- ดู province / amphoe / tambon
- ใช้ attribute join กับ pollution records

ใน Notebook นี้จะใช้เพียง **Pandas merge**

ยังไม่ใช้ spatial join

In [ ]:
# CELL 6 — Read station metadata

station_meta = pd.read_csv(
    PCD_STATION_CSV,
    dtype={
        "station_id":
            "string"
    },
)

print(
    "Station metadata shape:",
    station_meta.shape
)

print(
    "\nColumns:"
)

print(
    list(
        station_meta.columns
    )
)

display(
    station_meta.head(
        10
    )
)

In [ ]:
# CELL 7 — Basic station-metadata QC

required_station_fields = [
    "station_id",
    "latitude",
    "longitude",
]

missing_station_fields = [
    field
    for field
    in required_station_fields
    if field
    not in station_meta.columns
]

if missing_station_fields:

    raise KeyError(
        "Station metadata is missing: "
        + ", ".join(
            missing_station_fields
        )
    )


# Preserve the source representation for teaching/provenance.
station_meta[
    "station_id_source"
] = station_meta[
    "station_id"
].astype(
    "string"
)


# Canonical station key:
# lowercase + strip whitespace
#
# This is important because the PCD DATA sheet may use
# station columns such as 02T while another source may store 02t.
station_meta[
    "station_id"
] = (
    station_meta[
        "station_id"
    ]
    .astype(
        "string"
    )
    .str.strip()
    .str.lower()
)


station_metadata_qc = pd.DataFrame([
    {
        "check":
            "station_rows",

        "value":
            len(
                station_meta
            ),
    },

    {
        "check":
            "unique_station_id",

        "value":
            station_meta[
                "station_id"
            ].nunique(),
    },

    {
        "check":
            "duplicate_station_id_rows",

        "value":
            int(
                station_meta[
                    "station_id"
                ]
                .duplicated(
                    keep=False
                )
                .sum()
            ),
    },

    {
        "check":
            "missing_latitude",

        "value":
            int(
                station_meta[
                    "latitude"
                ].isna().sum()
            ),
    },

    {
        "check":
            "missing_longitude",

        "value":
            int(
                station_meta[
                    "longitude"
                ].isna().sum()
            ),
    },
])


display(
    station_metadata_qc
)


# 2.4 Excel Is Not Yet a DataFrame

Excel workbook อาจมี:

```text
1 workbook
    ↓
multiple sheets
    ↓
different columns
    ↓
different data types
```

ดังนั้นก่อน `concat()` เราต้องสำรวจโครงสร้างก่อน

> **Never concatenate first and inspect later.**

In [ ]:
# CELL 8 — Workbook inventory

workbook_inventory_rows = []

for year, path in EXCEL_FILES.items():

    workbook = pd.ExcelFile(
        path
    )

    for sheet in workbook.sheet_names:

        sample = pd.read_excel(
            path,
            sheet_name=sheet,
            nrows=10,
        )

        workbook_inventory_rows.append({
            "year":
                year,

            "filename":
                path.name,

            "sheet":
                sheet,

            "sample_rows":
                len(
                    sample
                ),

            "n_columns":
                len(
                    sample.columns
                ),

            "columns":
                " | ".join(
                    [
                        str(
                            c
                        )
                        for c
                        in sample.columns
                    ]
                ),
        })


workbook_inventory = pd.DataFrame(
    workbook_inventory_rows
)

display(
    workbook_inventory
)

# 2.5 Column Names Are Data Too

ข้อมูลหลายปีอาจเขียนชื่อ field ต่างกัน เช่น:

```text
stationID
station_id
StationID

PM25
PM2.5
PM_25

DATETIMEDATA
date
datetime
```

เราจะใช้ **column normalization** เพื่อช่วยตรวจ

แต่จะยังเก็บชื่อเดิมไว้ใน schema report

หลักการสำคัญ:

> Harmonization ต้องโปร่งใส  
> ไม่ใช่ rename แล้วลืมว่า source เดิมชื่ออะไร

In [ ]:
# CELL 9 — Column normalization helpers

def normalize_column_name(
    name,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(
            name
        )
        .strip()
        .lower()
    )


DATE_ALIASES = [
    "datetime",
    "date",
    "timestamp",
    "datetimedata",
    "date_time",
    "datetime_data",
]


STATION_ALIASES = [
    "stationid",
    "station_id",
    "station",
    "stationcode",
    "station_code",
    "siteid",
    "site_id",
]


POLLUTANT_ALIASES = {
    "pm25": [
        "pm25",
        "pm2.5",
        "pm2_5",
        "pm_25",
        "pm2p5",
    ],

    "pm10": [
        "pm10",
        "pm_10",
    ],

    "o3": [
        "o3",
        "ozone",
    ],

    "co": [
        "co",
        "carbonmonoxide",
    ],

    "no2": [
        "no2",
        "nitrogendioxide",
    ],

    "so2": [
        "so2",
        "sulfurdioxide",
        "sulphurdioxide",
    ],

    "aqi": [
        "aqi",
    ],
}


def detect_column(
    columns,
    aliases,
):
    normalized = {
        normalize_column_name(
            col
        ): col
        for col
        in columns
    }

    for alias in aliases:

        key = normalize_column_name(
            alias
        )

        if key in normalized:

            return normalized[
                key
            ]

    return None

# 2.6 Detect the Real Structure of Each Excel Sheet

ข้อมูล PCD/Air4Thai รายวันชุดนี้มีโครงสร้างที่สำคัญมาก:

## Layout A — PCD Daily Matrix

ใน `DATA` sheet:

```text
Date       02T   03T   05T   99T   ...
2024-01-01  21    18    25    30
2024-01-02  24    20    28    32
...
```

ดังนั้น:

```text
row       = วัน
column    = สถานี
cell      = PM2.5 เฉลี่ย 24 ชั่วโมง
```

ไม่มี column ชื่อ `station_id` อยู่ในตารางนี้

เราต้องแปลง:

```text
Date × Station Matrix
        ↓ melt()
date | station_id | pm25
```

นี่เป็นตัวอย่างจริงของการเปลี่ยนข้อมูลจาก **wide matrix → tidy/long observations**

---

## Layout B — Tidy Table

Notebook ยังรองรับกรณีที่ source อื่นมี:

```text
datetime | station_id | pm25 | pm10 | ...
```

โดยตรวจ explicit date/station/pollutant columns

---

## Detection Principle

Notebook จะไม่ดูเพียงชื่อ column แต่ตรวจร่วมกัน:

- sheet name
- first-column date parseability
- explicit station field
- pollutant fields
- จำนวน station-like value columns

ทุก sheet ถูกบันทึกว่า:

```text
PCD_DAILY_MATRIX
TIDY_TABLE
REJECTED
```

พร้อม diagnostic information

In [ ]:
# CELL 10 — Detect PCD matrix and tidy-table layouts

def candidate_matrix_station_columns(
    sample,
    first_column,
):
    """
    Identify plausible station columns in a PCD daily matrix.

    A candidate column:
    - is not the first/date column
    - has a non-empty name
    - contains at least one numeric observation in the sample

    We intentionally do not require one rigid station-ID regex because
    Air4Thai station IDs may contain letters and digits in different forms.
    """

    candidates = []

    for column in sample.columns:

        if column == first_column:
            continue

        name = str(
            column
        ).strip()

        if (
            name == ""
            or name.lower().startswith(
                "unnamed"
            )
        ):
            continue

        numeric = pd.to_numeric(
            sample[
                column
            ],
            errors="coerce",
        )

        if numeric.notna().any():

            candidates.append(
                column
            )

    return candidates


sheet_detection_rows = []


for year, path in EXCEL_FILES.items():

    workbook = pd.ExcelFile(
        path
    )

    for sheet in workbook.sheet_names:

        sample = pd.read_excel(
            path,
            sheet_name=sheet,
            nrows=250,
        )

        # --------------------------------------------------------------
        # A. Explicit tidy-table detection
        # --------------------------------------------------------------
        date_col_tidy = detect_column(
            sample.columns,
            DATE_ALIASES,
        )

        station_col_tidy = detect_column(
            sample.columns,
            STATION_ALIASES,
        )

        pollutants_found = {}

        for canonical, aliases in POLLUTANT_ALIASES.items():

            source_col = detect_column(
                sample.columns,
                aliases,
            )

            if source_col is not None:

                pollutants_found[
                    canonical
                ] = source_col


        tidy_detected = (
            date_col_tidy is not None
            and station_col_tidy is not None
            and len(
                pollutants_found
            ) > 0
        )


        # --------------------------------------------------------------
        # B. PCD daily matrix detection
        # --------------------------------------------------------------
        matrix_detected = False
        matrix_date_col = None
        matrix_station_cols = []
        first_date_valid_fraction = np.nan

        if sample.shape[
            1
        ] >= 2:

            first_col = sample.columns[
                0
            ]

            parsed_first = pd.to_datetime(
                sample[
                    first_col
                ],
                errors="coerce",
            )

            non_null_first_n = int(
                sample[
                    first_col
                ]
                .notna()
                .sum()
            )

            if non_null_first_n > 0:

                first_date_valid_fraction = (
                    parsed_first.notna().sum()
                    / non_null_first_n
                )

            else:

                first_date_valid_fraction = 0.0


            matrix_station_cols = (
                candidate_matrix_station_columns(
                    sample,
                    first_col,
                )
            )


            # DATA is the known Air4Thai archive sheet.
            # Date parseability + numeric station columns provide an
            # additional structural check rather than trusting the name alone.
            matrix_detected = (
                str(
                    sheet
                )
                .strip()
                .upper()
                == "DATA"
            ) and (
                first_date_valid_fraction
                >= 0.50
            ) and (
                len(
                    matrix_station_cols
                )
                >= 1
            )

            if matrix_detected:

                matrix_date_col = first_col


        # Prefer the known PCD matrix if detected.
        if matrix_detected:

            accepted = True
            layout_type = (
                "PCD_DAILY_MATRIX"
            )

            detected_date_col = (
                matrix_date_col
            )

            detected_station_col = None

            detected_pollutants = (
                "pm25"
            )

            pollutant_mapping_text = (
                "{'pm25': 'matrix cell values'}"
            )

            rejection_reason = ""

        elif tidy_detected:

            accepted = True
            layout_type = (
                "TIDY_TABLE"
            )

            detected_date_col = (
                date_col_tidy
            )

            detected_station_col = (
                station_col_tidy
            )

            detected_pollutants = (
                " | ".join(
                    sorted(
                        pollutants_found.keys()
                    )
                )
            )

            pollutant_mapping_text = str(
                pollutants_found
            )

            rejection_reason = ""

        else:

            accepted = False
            layout_type = (
                "REJECTED"
            )

            detected_date_col = (
                date_col_tidy
            )

            detected_station_col = (
                station_col_tidy
            )

            detected_pollutants = (
                " | ".join(
                    sorted(
                        pollutants_found.keys()
                    )
                )
            )

            pollutant_mapping_text = str(
                pollutants_found
            )

            reasons = []

            if (
                str(
                    sheet
                )
                .strip()
                .upper()
                == "DATA"
            ):

                if (
                    first_date_valid_fraction
                    < 0.50
                ):

                    reasons.append(
                        "DATA_first_column_not_date_like"
                    )

                if len(
                    matrix_station_cols
                ) == 0:

                    reasons.append(
                        "DATA_no_numeric_station_columns"
                    )

            else:

                reasons.append(
                    "not_DATA_matrix"
                )

            if not tidy_detected:

                reasons.append(
                    "not_tidy_airquality_table"
                )

            rejection_reason = (
                " | ".join(
                    reasons
                )
            )


        sheet_detection_rows.append({
            "year":
                year,

            "sheet":
                sheet,

            "accepted":
                accepted,

            "layout_type":
                layout_type,

            "date_source_column":
                detected_date_col,

            "station_source_column":
                detected_station_col,

            "matrix_station_columns_n":
                len(
                    matrix_station_cols
                ),

            "first_column_date_valid_fraction":
                first_date_valid_fraction,

            "pollutants_found":
                detected_pollutants,

            "pollutant_source_mapping":
                pollutant_mapping_text,

            "reason_if_rejected":
                rejection_reason,
        })


sheet_detection = pd.DataFrame(
    sheet_detection_rows
)


display(
    sheet_detection
)


print(
    "\nAccepted layouts:"
)

display(
    sheet_detection[
        sheet_detection[
            "accepted"
        ]
    ][
        [
            "year",
            "sheet",
            "layout_type",
            "date_source_column",
            "matrix_station_columns_n",
            "pollutants_found",
        ]
    ]
)


# Checkpoint 1 — เข้าใจโครงสร้าง PCD ก่อนวิเคราะห์

ดู `sheet_detection`

สำหรับ Air4Thai daily PM₂.₅ archive ที่ถูกต้อง เราคาดว่าจะพบ:

```text
sheet        = DATA
layout_type  = PCD_DAILY_MATRIX
pollutant    = pm25
```

และจำนวน `matrix_station_columns_n` ควรมากกว่า 0

ส่วน sheet เช่น:

```text
รายละเอียดจุดตรวจวัด
```

มีหน้าที่เป็น metadata ไม่ใช่ concentration table จึงไม่ควรถูกอ่านเหมือน `DATA`

---

## คำถามสำหรับนิสิต

1. ทำไม `DATA` sheet จึงไม่มี `station_id` column?
2. station IDs อยู่ที่ส่วนใดของ spreadsheet?
3. ค่า PM₂.₅ อยู่ใน row, column หรือ cell?
4. ทำไมจึงต้องใช้ `melt()`?
5. matrix format กับ tidy format ต่างกันอย่างไร?

นี่เป็นบทเรียนสำคัญมาก เพราะ environmental datasets จำนวนมากไม่ได้มาในรูป tidy table ตั้งแต่ต้น

# 2.7 Harmonize the Real PCD Workbook Structure

สำหรับ `PCD_DAILY_MATRIX`:

```text
DATA sheet
Date | 02T | 03T | 05T | ...
```

จะถูกแปลงเป็น:

```text
datetime | station_id | pm25
```

ด้วย `melt()`

จากนั้นเพิ่ม canonical fields:

```text
date
source_year
source_file
source_sheet
source_layout
```

สำหรับ pollutant อื่นที่ไม่มีใน workbook นี้จะเก็บเป็น `NaN`

---

## สำคัญ: ข้อมูลนี้เป็น Daily PM₂.₅ Archive

ค่าที่อ่านจาก matrix คือ **PM₂.₅ เฉลี่ย 24 ชั่วโมงที่ PCD เผยแพร่แล้ว**

ดังนั้น Notebook นี้จะไม่พยายามสร้าง daily mean จาก hourly observations ที่ไม่มีอยู่ในไฟล์

เมื่อภายหลังเรา `groupby(station_id, date)` อีกครั้ง จุดประสงค์คือ:

- ตรวจ duplicate
- ยืนยัน one station-day structure
- สร้าง canonical daily table

ไม่ใช่การทำ hourly completeness QA

In [ ]:
# CELL 11 — Harmonization function for PCD matrix and tidy tables

CANONICAL_POLLUTANTS = list(
    POLLUTANT_ALIASES.keys()
)


def harmonize_airquality_sheet(
    path,
    year,
    sheet,
    layout_type,
):
    raw = pd.read_excel(
        path,
        sheet_name=sheet,
    )


    # ==============================================================
    # Layout A — PCD DAILY MATRIX
    # ==============================================================
    if (
        layout_type
        == "PCD_DAILY_MATRIX"
    ):

        if raw.shape[
            1
        ] < 2:

            raise ValueError(
                f"{year}/{sheet}: DATA matrix has fewer than 2 columns."
            )


        date_col = raw.columns[
            0
        ]

        raw[
            date_col
        ] = pd.to_datetime(
            raw[
                date_col
            ],
            errors="coerce",
        )


        # Footer/notes/non-date rows are excluded from the observation table.
        valid_date_rows = raw[
            date_col
        ].notna()

        matrix = raw.loc[
            valid_date_rows
        ].copy()


        # Station columns are all non-date columns that contain
        # at least one numeric value after date rows are selected.
        station_columns = []

        for column in matrix.columns[
            1:
        ]:

            name = str(
                column
            ).strip()

            if (
                name == ""
                or name.lower().startswith(
                    "unnamed"
                )
            ):

                continue

            numeric = pd.to_numeric(
                matrix[
                    column
                ],
                errors="coerce",
            )

            if numeric.notna().any():

                station_columns.append(
                    column
                )


        if not station_columns:

            raise ValueError(
                f"{year}/{sheet}: no usable station columns found."
            )


        long = matrix[
            [
                date_col
            ]
            + station_columns
        ].melt(
            id_vars=[
                date_col
            ],
            value_vars=
                station_columns,
            var_name=
                "station_id_source",
            value_name=
                "pm25_source",
        )


        out = pd.DataFrame({
            "datetime_raw":
                long[
                    date_col
                ],

            "datetime":
                pd.to_datetime(
                    long[
                        date_col
                    ],
                    errors="coerce",
                ),

            "station_id_source":
                long[
                    "station_id_source"
                ]
                .astype(
                    "string"
                )
                .str.strip(),

            "station_id":
                long[
                    "station_id_source"
                ]
                .astype(
                    "string"
                )
                .str.strip()
                .str.lower(),

            "pm25":
                pd.to_numeric(
                    long[
                        "pm25_source"
                    ],
                    errors="coerce",
                ),
        })


        # The canonical table keeps expected pollutant columns so that
        # downstream code has a stable schema.
        for pollutant in CANONICAL_POLLUTANTS:

            if pollutant not in out.columns:

                out[
                    pollutant
                ] = np.nan


        pollutant_mapping = {
            "pm25":
                (
                    "DATA matrix cell values; "
                    "columns are station IDs"
                )
        }

        source_date_column = str(
            date_col
        )

        source_station_column = (
            "COLUMN HEADERS"
        )


    # ==============================================================
    # Layout B — explicit tidy table
    # ==============================================================
    elif (
        layout_type
        == "TIDY_TABLE"
    ):

        date_col = detect_column(
            raw.columns,
            DATE_ALIASES,
        )

        station_col = detect_column(
            raw.columns,
            STATION_ALIASES,
        )

        if date_col is None:

            raise ValueError(
                f"{year}/{sheet}: date column not found."
            )

        if station_col is None:

            raise ValueError(
                f"{year}/{sheet}: station column not found."
            )


        out = pd.DataFrame(
            index=raw.index
        )

        out[
            "datetime_raw"
        ] = raw[
            date_col
        ]

        out[
            "datetime"
        ] = pd.to_datetime(
            raw[
                date_col
            ],
            errors="coerce",
        )

        out[
            "station_id_source"
        ] = (
            raw[
                station_col
            ]
            .astype(
                "string"
            )
            .str.strip()
        )

        out[
            "station_id"
        ] = (
            out[
                "station_id_source"
            ]
            .str.lower()
        )


        pollutant_mapping = {}

        for canonical, aliases in POLLUTANT_ALIASES.items():

            source_col = detect_column(
                raw.columns,
                aliases,
            )

            pollutant_mapping[
                canonical
            ] = source_col

            if source_col is None:

                out[
                    canonical
                ] = np.nan

            else:

                out[
                    canonical
                ] = pd.to_numeric(
                    raw[
                        source_col
                    ],
                    errors="coerce",
                )


        source_date_column = str(
            date_col
        )

        source_station_column = str(
            station_col
        )


    else:

        raise ValueError(
            f"Unsupported layout_type: {layout_type}"
        )


    # ==============================================================
    # Common provenance fields
    # ==============================================================
    out[
        "source_year"
    ] = year

    out[
        "source_file"
    ] = path.name

    out[
        "source_sheet"
    ] = sheet

    out[
        "source_layout"
    ] = layout_type

    out[
        "source_date_column"
    ] = source_date_column

    out[
        "source_station_column"
    ] = source_station_column


    return (
        out,
        pollutant_mapping,
    )


In [ ]:
# CELL 12 — Harmonize all accepted sheets

harmonized_tables = []
mapping_rows = []
harmonization_report_rows = []


accepted_sheets = (
    sheet_detection[
        sheet_detection[
            "accepted"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "Accepted sheets to harmonize:",
    len(
        accepted_sheets
    )
)


display(
    accepted_sheets[
        [
            "year",
            "sheet",
            "layout_type",
            "pollutants_found",
        ]
    ]
)


for _, row in accepted_sheets.iterrows():

    year = int(
        row[
            "year"
        ]
    )

    sheet = row[
        "sheet"
    ]

    layout_type = row[
        "layout_type"
    ]

    path = EXCEL_FILES[
        year
    ]


    try:

        harmonized, mapping = (
            harmonize_airquality_sheet(
                path=path,
                year=year,
                sheet=sheet,
                layout_type=
                    layout_type,
            )
        )


        harmonized_tables.append(
            harmonized
        )


        harmonization_report_rows.append({
            "year":
                year,

            "sheet":
                sheet,

            "layout_type":
                layout_type,

            "status":
                "PASS",

            "output_rows":
                len(
                    harmonized
                ),

            "station_n":
                harmonized[
                    "station_id"
                ].nunique(),

            "pm25_non_missing_n":
                int(
                    harmonized[
                        "pm25"
                    ].notna().sum()
                ),
        })


        for canonical in CANONICAL_POLLUTANTS:

            mapping_rows.append({
                "year":
                    year,

                "sheet":
                    sheet,

                "layout_type":
                    layout_type,

                "canonical_field":
                    canonical,

                "source_field":
                    mapping.get(
                        canonical
                    ),
            })


    except Exception as exc:

        harmonization_report_rows.append({
            "year":
                year,

            "sheet":
                sheet,

            "layout_type":
                layout_type,

            "status":
                "FAIL",

            "error":
                str(
                    exc
                ),
        })


harmonization_report = pd.DataFrame(
    harmonization_report_rows
)


display(
    harmonization_report
)


if not harmonized_tables:

    raise RuntimeError(
        "No usable PCD air-quality sheets could be harmonized. "
        "Inspect sheet_detection and harmonization_report."
    )


air_wide = pd.concat(
    harmonized_tables,
    ignore_index=True,
    sort=False,
)


pollutant_column_mapping = pd.DataFrame(
    mapping_rows
)


print(
    "\nCombined canonical observation table:",
    air_wide.shape
)

print(
    "Source layouts:"
)

print(
    air_wide[
        "source_layout"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nPM2.5 non-missing records:",
    int(
        air_wide[
            "pm25"
        ].notna().sum()
    )
)


print(
    "Stations represented:",
    air_wide[
        "station_id"
    ].nunique()
)


display(
    pollutant_column_mapping
)

display(
    air_wide.head(
        10
    )
)


## สิ่งที่เกิดขึ้นกับ `DATA` sheet

ตัวอย่าง source matrix:

```text
Date       02T   03T   05T
2024-01-01  21    18    25
2024-01-02  24    20    28
```

หลัง harmonization:

```text
datetime    station_id    pm25
2024-01-01  02t           21
2024-01-01  03t           18
2024-01-01  05t           25
2024-01-02  02t           24
...
```

นี่คือแนวคิด **tidy data**:

> หนึ่ง row = หนึ่ง observation ของหนึ่งสถานีในหนึ่งเวลา

# 2.8 Datetime Is a Scientific Variable

วันเวลาไม่ควรถูกเก็บเป็น string ตลอดการวิเคราะห์

จาก `datetime` เราสร้าง:

```text
date
year
month
month_name
day_of_week
is_weekend
```

แต่ต้องระวังว่า:

> การมี field `year` ในชื่อไฟล์ไม่ได้รับประกันว่า datetime ข้างในอยู่ในปีนั้นทั้งหมด

ดังนั้นเราจะตรวจด้วย

In [ ]:
# CELL 13 — Derive temporal fields

air_wide[
    "date"
] = (
    air_wide[
        "datetime"
    ]
    .dt.floor(
        "D"
    )
)

air_wide[
    "year"
] = air_wide[
    "datetime"
].dt.year

air_wide[
    "month"
] = air_wide[
    "datetime"
].dt.month

air_wide[
    "month_name"
] = air_wide[
    "datetime"
].dt.month_name()

air_wide[
    "day_of_week"
] = air_wide[
    "datetime"
].dt.day_name()

air_wide[
    "is_weekend"
] = (
    air_wide[
        "datetime"
    ]
    .dt.dayofweek
    .isin(
        [
            5,
            6,
        ]
    )
)

In [ ]:
# CELL 14 — Datetime QC

air_wide[
    "datetime_valid"
] = air_wide[
    "datetime"
].notna()

air_wide[
    "source_year_matches_datetime"
] = (
    air_wide[
        "year"
    ]
    .eq(
        air_wide[
            "source_year"
        ]
    )
    |
    air_wide[
        "year"
    ].isna()
)


datetime_qc = pd.DataFrame([
    {
        "check":
            "total_rows",

        "value":
            len(
                air_wide
            ),
    },

    {
        "check":
            "invalid_datetime_rows",

        "value":
            int(
                (
                    ~air_wide[
                        "datetime_valid"
                    ]
                ).sum()
            ),
    },

    {
        "check":
            "source_year_datetime_mismatch_rows",

        "value":
            int(
                (
                    ~air_wide[
                        "source_year_matches_datetime"
                    ]
                ).sum()
            ),
    },

    {
        "check":
            "datetime_min",

        "value":
            air_wide[
                "datetime"
            ].min(),
    },

    {
        "check":
            "datetime_max",

        "value":
            air_wide[
                "datetime"
            ].max(),
    },
])


display(
    datetime_qc
)

# 2.9 Station IDs and Attribute Join

ค่ามลพิษมี `station_id`

station metadata ก็มี `station_id`

จึงสามารถเชื่อมกันด้วย:

```python
pd.merge()
```

นี่เรียกว่า **attribute join**

ยังไม่ใช่ spatial join

แนวคิด:

```text
Pollution table
station_id = 05T
        +
Station metadata
station_id = 05T
        ↓
Pollution + station name + province + coordinates
```

In [ ]:
# CELL 15 — Station ID QC before merge

observed_station_ids = set(
    air_wide[
        "station_id"
    ]
    .dropna()
    .astype(str)
)

metadata_station_ids = set(
    station_meta[
        "station_id"
    ]
    .dropna()
    .astype(str)
)


station_ids_not_in_metadata = sorted(
    observed_station_ids
    - metadata_station_ids
)

metadata_ids_not_in_observations = sorted(
    metadata_station_ids
    - observed_station_ids
)


station_id_qc = pd.DataFrame([
    {
        "check":
            "unique_station_ids_in_pollution",

        "value":
            len(
                observed_station_ids
            ),
    },

    {
        "check":
            "unique_station_ids_in_metadata",

        "value":
            len(
                metadata_station_ids
            ),
    },

    {
        "check":
            "pollution_station_ids_not_in_metadata",

        "value":
            len(
                station_ids_not_in_metadata
            ),
    },

    {
        "check":
            "metadata_ids_without_2021_2025_observations",

        "value":
            len(
                metadata_ids_not_in_observations
            ),
    },
])


display(
    station_id_qc
)


if station_ids_not_in_metadata:

    print(
        "Example station IDs not found in metadata:"
    )

    print(
        station_ids_not_in_metadata[
            :20
        ]
    )

In [ ]:
# CELL 16 — Merge station metadata

station_fields_for_merge = [
    field
    for field
    in [
        "station_id",
        "station_name_th",
        "station_name_en",
        "area_th",
        "area_en",
        "station_type",
        "latitude",
        "longitude",
        "province_code",
        "province_name_en",
        "province_name_th",
        "amphoe_code",
        "amphoe_name_en",
        "amphoe_name_th",
        "tambon_code",
        "tambon_name_en",
        "tambon_name_th",
    ]
    if field
    in station_meta.columns
]


air_wide = air_wide.merge(
    station_meta[
        station_fields_for_merge
    ],
    on="station_id",
    how="left",
    validate="many_to_one",
)


air_wide[
    "station_metadata_matched"
] = air_wide[
    "latitude"
].notna()


print(
    "Rows matched with station metadata:",
    int(
        air_wide[
            "station_metadata_matched"
        ].sum()
    ),
    "/",
    len(
        air_wide
    )
)

# 2.10 Missing Values

เราจะตรวจ missingness สองระดับ:

### ระดับ field

```text
PM2.5 missing กี่ %
PM10 missing กี่ %
...
```

### ระดับปี

```text
PM2.5 missing ใน 2021 เท่าไร?
PM2.5 missing ใน 2025 เท่าไร?
```

การเปรียบเทียบข้ามปีควรดู data availability ควบคู่กัน

In [ ]:
# CELL 17 — Pollutant missingness

pollutant_missing_rows = []

for pollutant in CANONICAL_POLLUTANTS:

    if pollutant not in air_wide.columns:
        continue

    pollutant_missing_rows.append({
        "pollutant":
            pollutant,

        "total_rows":
            len(
                air_wide
            ),

        "non_missing_n":
            int(
                air_wide[
                    pollutant
                ].notna().sum()
            ),

        "missing_n":
            int(
                air_wide[
                    pollutant
                ].isna().sum()
            ),

        "missing_pct":
            (
                air_wide[
                    pollutant
                ].isna().mean()
                * 100
            ),
    })


pollutant_missingness = pd.DataFrame(
    pollutant_missing_rows
)


display(
    pollutant_missingness
)

In [ ]:
# CELL 18 — PM2.5 missingness by source year

pm25_missing_by_year = (
    air_wide
    .groupby(
        "source_year",
        dropna=False,
    )
    .agg(
        rows=(
            "pm25",
            "size",
        ),

        pm25_non_missing_n=(
            "pm25",
            "count",
        ),

        station_n=(
            "station_id",
            "nunique",
        ),
    )
    .reset_index()
)


pm25_missing_by_year[
    "pm25_missing_n"
] = (
    pm25_missing_by_year[
        "rows"
    ]
    -
    pm25_missing_by_year[
        "pm25_non_missing_n"
    ]
)


pm25_missing_by_year[
    "pm25_missing_pct"
] = (
    pm25_missing_by_year[
        "pm25_missing_n"
    ]
    /
    pm25_missing_by_year[
        "rows"
    ]
    * 100
)


display(
    pm25_missing_by_year
)

# 2.11 Duplicate Records

Duplicate ต้องนิยามให้ชัดก่อน

สำหรับข้อมูล station-time เราจะตรวจ:

```text
station_id + datetime
```

ถ้ามีมากกว่า 1 record ต่อ station-time:

อาจเกิดจาก:

- duplicate จริง
- หลาย sheet ซ้ำกัน
- หลาย sensor/channel
- ingestion ซ้ำ
- source structure ที่มีเหตุผล

ดังนั้น Notebook จะ **flag ก่อน ไม่ delete ก่อน**

In [ ]:
# CELL 19 — Duplicate station-time QC

air_wide[
    "duplicate_station_datetime"
] = (
    air_wide
    .duplicated(
        subset=[
            "station_id",
            "datetime",
        ],
        keep=False,
    )
)


duplicate_qc = pd.DataFrame([
    {
        "check":
            "duplicate_station_datetime_rows",

        "value":
            int(
                air_wide[
                    "duplicate_station_datetime"
                ].sum()
            ),
    },

    {
        "check":
            "unique_station_datetime_pairs",

        "value":
            air_wide[
                [
                    "station_id",
                    "datetime",
                ]
            ]
            .drop_duplicates()
            .shape[
                0
            ],
    },
])


display(
    duplicate_qc
)


duplicate_preview = (
    air_wide[
        air_wide[
            "duplicate_station_datetime"
        ]
    ]
    .sort_values(
        [
            "station_id",
            "datetime",
        ]
    )
    .head(
        20
    )
)


if len(
    duplicate_preview
) > 0:

    display(
        duplicate_preview
    )

# 2.12 QC Flags for Pollutant Concentrations

สำหรับ concentration pollutants เช่น:

```text
PM2.5
PM10
O3
CO
NO2
SO2
```

ค่าติดลบโดยทั่วไปไม่ใช่ concentration ที่มีความหมายทางกายภาพ

แต่แทนที่จะเขียน:

```python
df = df[df["pm25"] >= 0]
```

แล้วทำข้อมูลหายทันที เราจะสร้าง:

```text
pm25_negative_flag
pm25_valid_basic
```

แล้วเก็บ raw values ไว้

นี่คือหลัก:

> **Flag first, filter later**

In [ ]:
# CELL 20 — Basic pollutant QC flags

CONCENTRATION_FIELDS = [
    "pm25",
    "pm10",
    "o3",
    "co",
    "no2",
    "so2",
]


for pollutant in CONCENTRATION_FIELDS:

    if pollutant not in air_wide.columns:
        continue

    air_wide[
        f"{pollutant}_negative_flag"
    ] = (
        air_wide[
            pollutant
        ]
        < 0
    )

    air_wide[
        f"{pollutant}_valid_basic"
    ] = (
        air_wide[
            pollutant
        ].notna()
        &
        (
            air_wide[
                pollutant
            ]
            >= 0
        )
    )

In [ ]:
# CELL 21 — Basic QC summary by pollutant

basic_qc_rows = []

for pollutant in CONCENTRATION_FIELDS:

    if pollutant not in air_wide.columns:
        continue

    basic_qc_rows.append({
        "pollutant":
            pollutant,

        "non_missing_n":
            int(
                air_wide[
                    pollutant
                ].notna().sum()
            ),

        "negative_n":
            int(
                air_wide[
                    f"{pollutant}_negative_flag"
                ].sum()
            ),

        "basic_valid_n":
            int(
                air_wide[
                    f"{pollutant}_valid_basic"
                ].sum()
            ),

        "minimum_observed":
            air_wide[
                pollutant
            ].min(),

        "maximum_observed":
            air_wide[
                pollutant
            ].max(),
    })


basic_pollutant_qc = pd.DataFrame(
    basic_qc_rows
)

display(
    basic_pollutant_qc
)

## เรื่อง “ค่าสูงผิดปกติ”

Notebook นี้จะ **ไม่กำหนด upper cutoff แบบตายตัว**

เพราะ threshold ที่เหมาะสมขึ้นกับ:

- pollutant
- averaging period
- instrument/data product
- station context
- scientific question

ดังนั้น maximum value จะถูก **รายงาน** แต่ไม่ถูกลบอัตโนมัติ

ในงานวิจัยจริงควรตรวจ:

```text
neighboring stations
meteorology
fire/burning information
instrument status
temporal continuity
source documentation
```

ก่อนตัดสินว่าเป็น error

# 2.13 Wide Format → Long Format

Wide format:

| datetime | station | pm25 | pm10 | o3 |
|---|---|---:|---:|---:|

Long format:

| datetime | station | pollutant | value |
|---|---|---|---:|

Long format เหมาะกับ:

- `groupby`
- plotting
- comparing pollutants
- tidy-data workflow

เราจะเก็บ **ทั้ง wide และ long**

In [ ]:
# CELL 22 — Create long-format pollutant table

id_columns = [
    field
    for field
    in [
        "datetime",
        "date",
        "year",
        "month",
        "month_name",
        "day_of_week",
        "is_weekend",
        "station_id",
        "station_id_source",
        "station_name_th",
        "station_name_en",
        "province_name_en",
        "province_name_th",
        "latitude",
        "longitude",
        "source_year",
        "source_file",
        "source_sheet",
        "source_layout",
    ]
    if field
    in air_wide.columns
]


# Only melt pollutants that actually contain observations.
# The current PCD daily archive is a PM2.5 dataset, so this
# normally prevents creating millions of unnecessary all-NaN
# rows for pollutants that are absent.
pollutant_value_columns = [
    field
    for field
    in CONCENTRATION_FIELDS
    if (
        field
        in air_wide.columns
        and air_wide[
            field
        ].notna().any()
    )
]


if not pollutant_value_columns:

    raise RuntimeError(
        "No concentration pollutant contains usable observations."
    )


print(
    "Pollutants present in this archive:",
    pollutant_value_columns
)


air_long = air_wide.melt(
    id_vars=id_columns,
    value_vars=pollutant_value_columns,
    var_name="pollutant",
    value_name="value_raw",
)


air_long[
    "negative_flag"
] = (
    air_long[
        "value_raw"
    ]
    < 0
)


air_long[
    "valid_basic"
] = (
    air_long[
        "value_raw"
    ].notna()
    &
    (
        air_long[
            "value_raw"
        ]
        >= 0
    )
)


air_long[
    "value"
] = air_long[
    "value_raw"
].where(
    air_long[
        "valid_basic"
    ]
)


print(
    "Canonical observation table:",
    air_wide.shape
)

print(
    "Long pollutant table:",
    air_long.shape
)


display(
    air_long.head(
        10
    )
)


# 2.14 Confirm the Temporal Resolution

ข้อมูลชุดนี้มาจาก Air4Thai **รายวัน — PM₂.₅ เฉลี่ย 24 ชั่วโมง**

ดังนั้นโดยหลักแล้ว:

```text
1 station-date ≈ 1 published daily PM2.5 value
```

อย่างไรก็ตาม เราจะตรวจ temporal spacing จากข้อมูลจริงเพื่อยืนยันว่า:

```text
median interval ≈ 24 h
```

และตรวจว่ามี:

- duplicate station-date
- missing days
- irregular gaps

หรือไม่

---

## สำคัญ

เรา **ไม่ใช้ hourly completeness rule** กับไฟล์นี้ เพราะ source workbook ไม่ได้มี hourly observations ให้ตรวจ

Data completeness ที่เหมาะสมในขั้นต่อไปจึงอยู่ในระดับ:

```text
daily availability
station-month completeness
station-year completeness
```

In [ ]:
# CELL 23 — Estimate temporal interval by station

interval_rows = []

for station_id, group in (
    air_wide[
        [
            "station_id",
            "datetime",
        ]
    ]
    .dropna()
    .sort_values(
        "datetime"
    )
    .groupby(
        "station_id"
    )
):

    times = (
        group[
            "datetime"
        ]
        .drop_duplicates()
        .sort_values()
    )

    if len(
        times
    ) < 2:
        continue

    delta_hours = (
        times.diff()
        .dt.total_seconds()
        / 3600
    )

    interval_rows.append({
        "station_id":
            station_id,

        "n_unique_times":
            len(
                times
            ),

        "median_interval_hours":
            delta_hours.median(),

        "min_interval_hours":
            delta_hours.min(),

        "max_interval_hours":
            delta_hours.max(),
    })


temporal_resolution = pd.DataFrame(
    interval_rows
)


display(
    temporal_resolution.describe(
        include="all"
    )
)

# 2.15 Focus Variable: PM₂.₅

จากนี้ใช้ PM₂.₅ เป็นตัวอย่างหลัก

แต่ workflow เดียวกันสามารถประยุกต์กับ PM₁₀, O₃, NO₂ ฯลฯ ได้

เราจะสร้าง table ที่เก็บเฉพาะ:

```text
datetime valid
station valid
PM2.5 non-missing
PM2.5 >= 0
```

เรียกว่า:

```text
pm25_basic_valid
```

คำว่า `basic` สำคัญ เพราะยังไม่ใช่ instrument-level QA/QC

In [ ]:
# CELL 24 — PM2.5 basic-valid table

if (
    "pm25"
    not in air_wide.columns
):

    raise KeyError(
        "PM2.5 canonical field was not found "
        "in the combined PCD dataset."
    )


pm25_data = air_wide[
    air_wide[
        "datetime"
    ].notna()
    &
    air_wide[
        "station_id"
    ].notna()
    &
    air_wide[
        "pm25_valid_basic"
    ]
].copy()


print(
    "PM2.5 basic-valid observations:",
    len(
        pm25_data
    )
)

print(
    "Stations:",
    pm25_data[
        "station_id"
    ].nunique()
)

print(
    "Date range:",
    pm25_data[
        "datetime"
    ].min(),
    "to",
    pm25_data[
        "datetime"
    ].max()
)

# 2.16 Canonical Station-Day Table

ค่าจาก `DATA` sheet เป็น PM₂.₅ เฉลี่ย 24 ชั่วโมงที่เผยแพร่เป็นรายวันอยู่แล้ว

ดังนั้นการ `groupby(station_id, date)` ในขั้นนี้มีหน้าที่หลักเพื่อ:

1. ตรวจว่ามี duplicate station-date หรือไม่
2. รวม accidental repeated records หากมี
3. สร้าง canonical station-day table ที่ใช้รูปแบบเดียวกันทุกปี

เรายังคงเก็บ:

```text
pm25_mean
pm25_median
pm25_min
pm25_max
n_obs
```

ใน dataset รายวันที่สะอาด `n_obs` ควรเป็น 1 ในเกือบทุก station-date

ถ้า `n_obs > 1` ควรย้อนกลับไปตรวจ duplicate/source sheets

In [ ]:
# CELL 25 — Daily PM2.5 summary

daily_group_fields = [
    "station_id",
    "date",
]


pm25_daily = (
    pm25_data
    .groupby(
        daily_group_fields,
        dropna=False,
    )
    .agg(
        pm25_mean=(
            "pm25",
            "mean",
        ),

        pm25_median=(
            "pm25",
            "median",
        ),

        pm25_min=(
            "pm25",
            "min",
        ),

        pm25_max=(
            "pm25",
            "max",
        ),

        n_obs=(
            "pm25",
            "count",
        ),
    )
    .reset_index()
)


station_merge_fields = [
    field
    for field
    in [
        "station_id",
        "station_name_th",
        "station_name_en",
        "province_name_en",
        "province_name_th",
        "latitude",
        "longitude",
    ]
    if field
    in station_meta.columns
]


pm25_daily = pm25_daily.merge(
    station_meta[
        station_merge_fields
    ],
    on="station_id",
    how="left",
    validate="many_to_one",
)


pm25_daily[
    "year"
] = pm25_daily[
    "date"
].dt.year

pm25_daily[
    "month"
] = pm25_daily[
    "date"
].dt.month


display(
    pm25_daily.head(
        10
    )
)

# 2.17 Station Coverage and Completeness

สมมติสถานีมีข้อมูลตั้งแต่:

```text
2024-01-01 ถึง 2024-12-31
```

จำนวนวันที่เป็นไปได้ประมาณ 366 วัน

ถ้ามี observation เพียง 100 วัน:

```text
coverage ≈ 27%
```

เราจะคำนวณ coverage ภายในช่วงที่แต่ละ station ปรากฏใน dataset

นี่เป็น **teaching completeness metric**

ยังไม่ใช่ข้อกำหนด regulatory completeness

In [ ]:
# CELL 26 — Station PM2.5 coverage

station_coverage = (
    pm25_daily
    .groupby(
        "station_id"
    )
    .agg(
        date_min=(
            "date",
            "min",
        ),

        date_max=(
            "date",
            "max",
        ),

        observed_days=(
            "date",
            "nunique",
        ),

        pm25_mean=(
            "pm25_mean",
            "mean",
        ),

        pm25_median=(
            "pm25_mean",
            "median",
        ),
    )
    .reset_index()
)


station_coverage[
    "possible_days_between_first_last"
] = (
    (
        station_coverage[
            "date_max"
        ]
        -
        station_coverage[
            "date_min"
        ]
    )
    .dt.days
    + 1
)


station_coverage[
    "coverage_pct_between_first_last"
] = (
    station_coverage[
        "observed_days"
    ]
    /
    station_coverage[
        "possible_days_between_first_last"
    ]
    * 100
)


station_coverage = station_coverage.merge(
    station_meta[
        station_merge_fields
    ],
    on="station_id",
    how="left",
    validate="one_to_one",
)


display(
    station_coverage.sort_values(
        "coverage_pct_between_first_last"
    ).head(
        20
    )
)

# 2.18 Annual and Monthly Descriptive Statistics

เราจะยังไม่ตีความว่า 2021–2025 เป็น “climatology”

ช่วง 5 ปีนี้เป็น:

> multi-year teaching period

การเรียกว่า climatology ต้องพิจารณาช่วงเวลาที่ยาวกว่าและหลักเกณฑ์ที่เหมาะสม

ในบทนี้จะใช้คำว่า:

```text
annual variation
monthly variation
2021–2025 teaching period
```

In [ ]:
# CELL 27 — Annual PM2.5 summary

annual_pm25 = (
    pm25_daily
    .groupby(
        "year"
    )
    .agg(
        station_n=(
            "station_id",
            "nunique",
        ),

        station_day_n=(
            "pm25_mean",
            "count",
        ),

        mean_pm25=(
            "pm25_mean",
            "mean",
        ),

        median_pm25=(
            "pm25_mean",
            "median",
        ),

        std_pm25=(
            "pm25_mean",
            "std",
        ),

        q25_pm25=(
            "pm25_mean",
            lambda x:
                x.quantile(
                    0.25
                ),
        ),

        q75_pm25=(
            "pm25_mean",
            lambda x:
                x.quantile(
                    0.75
                ),
        ),
    )
    .reset_index()
)


display(
    annual_pm25
)

In [ ]:
# CELL 28 — Monthly PM2.5 summary

monthly_pm25 = (
    pm25_daily
    .groupby(
        "month"
    )
    .agg(
        station_n=(
            "station_id",
            "nunique",
        ),

        station_day_n=(
            "pm25_mean",
            "count",
        ),

        mean_pm25=(
            "pm25_mean",
            "mean",
        ),

        median_pm25=(
            "pm25_mean",
            "median",
        ),

        q25_pm25=(
            "pm25_mean",
            lambda x:
                x.quantile(
                    0.25
                ),
        ),

        q75_pm25=(
            "pm25_mean",
            lambda x:
                x.quantile(
                    0.75
                ),
        ),
    )
    .reset_index()
)


display(
    monthly_pm25
)

# 2.19 Visualization 1 — Annual PM₂.₅

กราฟต้องอ่านร่วมกับ:

```text
station_n
station_day_n
```

เพราะค่าเฉลี่ยเปลี่ยนได้จากทั้ง:

1. atmospheric/environmental conditions
2. station network / data availability

In [ ]:
# CELL 29 — Annual PM2.5 plot

fig, ax = plt.subplots(
    figsize=(
        9,
        5,
    )
)

ax.plot(
    annual_pm25[
        "year"
    ],
    annual_pm25[
        "mean_pm25"
    ],
    marker="o",
    label="Mean",
)

ax.plot(
    annual_pm25[
        "year"
    ],
    annual_pm25[
        "median_pm25"
    ],
    marker="s",
    label="Median",
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "PM2.5 concentration"
)

ax.set_title(
    "Annual PM2.5 Variation — PCD Teaching Dataset"
)

ax.grid(
    alpha=0.25
)

ax.legend()

fig.tight_layout()

ANNUAL_FIG = (
    FIGURE_DIR
    / "02_PM25_annual_variation.png"
)

fig.savefig(
    ANNUAL_FIG,
    dpi=250,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    ANNUAL_FIG
)

# 2.20 Visualization 2 — Monthly Distribution

Boxplot แสดง:

```text
median
interquartile range
distribution spread
extreme observations
```

แต่จุดที่อยู่นอก whisker:

> ไม่ได้หมายความว่าเป็น error โดยอัตโนมัติ

In [ ]:
# CELL 30 — Monthly PM2.5 boxplot

monthly_groups = [
    pm25_daily.loc[
        pm25_daily[
            "month"
        ].eq(
            month
        ),
        "pm25_mean",
    ]
    .dropna()
    .values

    for month
    in range(
        1,
        13
    )
]


fig, ax = plt.subplots(
    figsize=(
        11,
        5.5,
    )
)

ax.boxplot(
    monthly_groups,
    labels=[
        str(
            month
        )
        for month
        in range(
            1,
            13
        )
    ],
    showfliers=True,
)

ax.set_xlabel(
    "Month"
)

ax.set_ylabel(
    "Daily PM2.5"
)

ax.set_title(
    "Monthly Distribution of Daily PM2.5"
)

ax.grid(
    axis="y",
    alpha=0.2,
)

fig.tight_layout()

MONTHLY_BOXPLOT = (
    FIGURE_DIR
    / "02_PM25_monthly_boxplot.png"
)

fig.savefig(
    MONTHLY_BOXPLOT,
    dpi=250,
    bbox_inches="tight",
)

plt.show()

# 2.21 Visualization 3 — Distribution

Histogram ช่วยตอบ:

- distribution สมมาตรหรือไม่?
- right-skewed หรือไม่?
- มี long tail หรือไม่?

ข้อมูลมลพิษจำนวนมากไม่จำเป็นต้องมี normal distribution

จึงไม่ควรสมมติ normality เพียงเพราะต้องการใช้ค่าเฉลี่ย

In [ ]:
# CELL 31 — PM2.5 histogram

fig, ax = plt.subplots(
    figsize=(
        8,
        5,
    )
)

ax.hist(
    pm25_daily[
        "pm25_mean"
    ].dropna(),
    bins=40,
)

ax.set_xlabel(
    "Daily PM2.5"
)

ax.set_ylabel(
    "Frequency"
)

ax.set_title(
    "Distribution of Daily PM2.5"
)

ax.grid(
    axis="y",
    alpha=0.2,
)

fig.tight_layout()

HIST_FIG = (
    FIGURE_DIR
    / "02_PM25_daily_histogram.png"
)

fig.savefig(
    HIST_FIG,
    dpi=250,
    bbox_inches="tight",
)

plt.show()

# 2.22 Compare Stations Without a Map

ก่อนเรียน GIS เราสามารถเปรียบเทียบสถานีด้วย table ได้

ตัวอย่างคำถาม:

- สถานีใดมีข้อมูลมากที่สุด?
- สถานีใดมี PM₂.₅ เฉลี่ยสูง?
- station ranking เปลี่ยนหรือไม่เมื่อใช้ median?
- station ที่มีค่าเฉลี่ยสูงมี completeness เพียงพอหรือไม่?

นี่เป็นเหตุผลที่ descriptive statistics ต้องมาพร้อม data coverage

In [ ]:
# CELL 32 — Station descriptive summary

station_pm25_summary = (
    pm25_daily
    .groupby(
        "station_id"
    )
    .agg(
        daily_n=(
            "pm25_mean",
            "count",
        ),

        mean_pm25=(
            "pm25_mean",
            "mean",
        ),

        median_pm25=(
            "pm25_mean",
            "median",
        ),

        std_pm25=(
            "pm25_mean",
            "std",
        ),

        q90_pm25=(
            "pm25_mean",
            lambda x:
                x.quantile(
                    0.90
                ),
        ),

        maximum_pm25=(
            "pm25_mean",
            "max",
        ),
    )
    .reset_index()
)


station_pm25_summary = (
    station_pm25_summary
    .merge(
        station_coverage[
            [
                "station_id",
                "coverage_pct_between_first_last",
            ]
        ],
        on="station_id",
        how="left",
        validate="one_to_one",
    )
)


station_pm25_summary = (
    station_pm25_summary
    .merge(
        station_meta[
            station_merge_fields
        ],
        on="station_id",
        how="left",
        validate="one_to_one",
    )
)


display(
    station_pm25_summary.sort_values(
        "mean_pm25",
        ascending=False,
    ).head(
        20
    )
)

# 2.23 Bridge to the Three-Province Case Study

Course นี้จะใช้พื้นที่:

```text
Saraburi
Lop Buri
Nakhon Nayok
```

เป็น case study ต่อเนื่องในหลายบท

ใน Notebook 02 เราเลือกจังหวัดจาก **attribute**
ที่ผู้สอนได้ spatially enrich ไว้ใน canonical station metadata

นี่ไม่ใช่ spatial analysis ใหม่

เพียงใช้ field:

```text
province_name_en
```

เพื่อ filter records

Notebook 03–04 จะอธิบายว่า field นี้ได้มาจาก geometry อย่างไร

In [ ]:
# CELL 33 — Inspect province names available in station metadata

if (
    "province_name_en"
    in station_meta.columns
):

    province_station_count = (
        station_meta
        .groupby(
            "province_name_en",
            dropna=False,
        )
        .agg(
            station_n=(
                "station_id",
                "nunique",
            )
        )
        .reset_index()
        .sort_values(
            "province_name_en"
        )
    )

    display(
        province_station_count
    )

else:

    print(
        "province_name_en is not available "
        "in station metadata."
    )

In [ ]:
# CELL 34 — Three-province PM2.5 subset

CASE_PROVINCES = [
    "Saraburi",
    "Lop Buri",
    "Nakhon Nayok",
]


if (
    "province_name_en"
    in pm25_daily.columns
):

    case_pm25_daily = (
        pm25_daily[
            pm25_daily[
                "province_name_en"
            ].isin(
                CASE_PROVINCES
            )
        ]
        .copy()
    )

    print(
        "Case-study station-days:",
        len(
            case_pm25_daily
        )
    )

    print(
        "Stations:",
        case_pm25_daily[
            "station_id"
        ].nunique()
    )

    display(
        (
            case_pm25_daily
            .groupby(
                "province_name_en"
            )
            .agg(
                station_n=(
                    "station_id",
                    "nunique",
                ),

                station_day_n=(
                    "pm25_mean",
                    "count",
                ),

                mean_pm25=(
                    "pm25_mean",
                    "mean",
                ),

                median_pm25=(
                    "pm25_mean",
                    "median",
                ),
            )
            .reset_index()
        )
    )

else:

    case_pm25_daily = pd.DataFrame()

    print(
        "Province attribute is unavailable."
    )

# 2.24 Pivot Table

`pivot_table()` เหมาะสำหรับสร้างตาราง:

```text
rows    = month
columns = year
values  = PM2.5
```

ซึ่งช่วยให้มอง seasonal pattern ข้ามปีได้ง่าย

In [ ]:
# CELL 35 — Year × month pivot table

year_month_pm25 = pd.pivot_table(
    pm25_daily,
    index="month",
    columns="year",
    values="pm25_mean",
    aggfunc="mean",
)


display(
    year_month_pm25
)

# 2.25 Save Analysis-Ready Outputs

Notebook นี้จะสร้าง output สำหรับบทต่อไป

## Wide harmonized table

```text
02_pcd_airquality_harmonized_2021_2025.csv.gz
```

ใช้เมื่ออยากเข้าถึง pollutants หลายตัว

## Long table

```text
02_pcd_airquality_long_2021_2025.csv.gz
```

เหมาะกับ tidy-data workflow

## PM₂.₅ daily

```text
02_pcd_pm25_daily_2021_2025.csv.gz
```

จะเป็นฐานสำหรับ Notebook 03–06

นอกจากนี้บันทึก QC summaries เพื่อ reproducibility

In [ ]:
# CELL 36 — Save analysis-ready data

AIR_WIDE_FILE = (
    OUTPUT_DIR
    / "02_pcd_airquality_harmonized_2021_2025.csv.gz"
)

AIR_LONG_FILE = (
    OUTPUT_DIR
    / "02_pcd_airquality_long_2021_2025.csv.gz"
)

PM25_DAILY_FILE = (
    OUTPUT_DIR
    / "02_pcd_pm25_daily_2021_2025.csv.gz"
)


air_wide.to_csv(
    AIR_WIDE_FILE,
    index=False,
    compression="gzip",
)

air_long.to_csv(
    AIR_LONG_FILE,
    index=False,
    compression="gzip",
)

pm25_daily.to_csv(
    PM25_DAILY_FILE,
    index=False,
    compression="gzip",
)


print(
    "Saved:"
)

for path in [
    AIR_WIDE_FILE,
    AIR_LONG_FILE,
    PM25_DAILY_FILE,
]:

    print(
        " -",
        path
    )

In [ ]:
# CELL 37 — Save QC and summary tables

OUTPUT_TABLES = {
    "02_workbook_inventory.csv":
        workbook_inventory,

    "02_sheet_detection.csv":
        sheet_detection,

    "02_pollutant_column_mapping.csv":
        pollutant_column_mapping,

    "02_harmonization_report.csv":
        harmonization_report,

    "02_datetime_QC.csv":
        datetime_qc,

    "02_station_ID_QC.csv":
        station_id_qc,

    "02_pollutant_missingness.csv":
        pollutant_missingness,

    "02_PM25_missingness_by_year.csv":
        pm25_missing_by_year,

    "02_duplicate_QC.csv":
        duplicate_qc,

    "02_basic_pollutant_QC.csv":
        basic_pollutant_qc,

    "02_temporal_resolution_by_station.csv":
        temporal_resolution,

    "02_station_PM25_coverage.csv":
        station_coverage,

    "02_annual_PM25_summary.csv":
        annual_pm25,

    "02_monthly_PM25_summary.csv":
        monthly_pm25,

    "02_station_PM25_summary.csv":
        station_pm25_summary,

    "02_year_month_PM25_pivot.csv":
        year_month_pm25.reset_index(),
}


for filename, table in OUTPUT_TABLES.items():

    path = (
        OUTPUT_DIR
        / filename
    )

    table.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "Saved:",
        filename
    )

# 2.26 Final Notebook Checkpoint

Notebook 02 ถือว่าพร้อมสำหรับบทถัดไปเมื่อ:

```text
Excel 2021–2025 readable
accepted data sheets > 0
datetime parsed
PM2.5 detected
station IDs available
station metadata merged
daily PM2.5 table created
outputs saved
```

แต่ `PASS` ในที่นี้หมายถึง:

> พร้อมสำหรับการเรียนและ exploratory analysis

ไม่ใช่การรับรองว่า raw observations ผ่าน regulatory QA/QC ทุกประการ

In [ ]:
# CELL 38 — Final Notebook 02 checkpoint

final_checks = pd.DataFrame([
    {
        "check":
            "Excel files available",

        "status":
            (
                "PASS"
                if len(
                    EXCEL_FILES
                ) == 5
                else "FAIL"
            ),

        "detail":
            f"{len(EXCEL_FILES)}/5",
    },

    {
        "check":
            "Accepted data sheets",

        "status":
            (
                "PASS"
                if len(
                    accepted_sheets
                ) > 0
                else "FAIL"
            ),

        "detail":
            str(
                len(
                    accepted_sheets
                )
            ),
    },

    {
        "check":
            "PCD DATA matrix detected",

        "status":
            (
                "PASS"
                if (
                    accepted_sheets[
                        "layout_type"
                    ]
                    .eq(
                        "PCD_DAILY_MATRIX"
                    )
                    .any()
                )
                else "CHECK"
            ),

        "detail":
            str(
                int(
                    accepted_sheets[
                        "layout_type"
                    ]
                    .eq(
                        "PCD_DAILY_MATRIX"
                    )
                    .sum()
                )
            ),
    },

    {
        "check":
            "Combined air-quality rows",

        "status":
            (
                "PASS"
                if len(
                    air_wide
                ) > 0
                else "FAIL"
            ),

        "detail":
            str(
                len(
                    air_wide
                )
            ),
    },

    {
        "check":
            "PM2.5 available",

        "status":
            (
                "PASS"
                if air_wide[
                    "pm25"
                ].notna().any()
                else "FAIL"
            ),

        "detail":
            str(
                int(
                    air_wide[
                        "pm25"
                    ].notna().sum()
                )
            ),
    },

    {
        "check":
            "Station metadata matched",

        "status":
            (
                "PASS"
                if air_wide[
                    "station_metadata_matched"
                ].any()
                else "FAIL"
            ),

        "detail":
            (
                f"{air_wide['station_metadata_matched'].mean()*100:.1f}%"
            ),
    },

    {
        "check":
            "Daily PM2.5 table",

        "status":
            (
                "PASS"
                if len(
                    pm25_daily
                ) > 0
                else "FAIL"
            ),

        "detail":
            str(
                len(
                    pm25_daily
                )
            ),
    },

    {
        "check":
            "Analysis-ready outputs saved",

        "status":
            (
                "PASS"
                if all(
                    path.exists()
                    for path
                    in [
                        AIR_WIDE_FILE,
                        AIR_LONG_FILE,
                        PM25_DAILY_FILE,
                    ]
                )
                else "FAIL"
            ),

        "detail":
            str(
                OUTPUT_DIR
            ),
    },
])


display(
    final_checks
)


if final_checks[
    "status"
].eq(
    "PASS"
).all():

    print(
        "\n===================================="
    )

    print(
        "NOTEBOOK 02 ANALYSIS DATA READY"
    )

    print(
        "===================================="
    )

else:

    print(
        "\nCHECK NOTEBOOK 02 QC RESULTS "
        "BEFORE CONTINUING"
    )

# แบบฝึกหัดท้ายบท

## Exercise 1 — DataFrame Fundamentals

เลือก Excel 1 ปี แล้วตอบ:

1. มีกี่ rows?
2. มีกี่ columns?
3. field ใดเป็น station identifier?
4. field ใดเป็นวันเวลา?
5. มี pollutant ใดบ้าง?
6. field ใดมี missing มากที่สุด?

---

## Exercise 2 — Schema Evolution

จาก:

```text
02_pollutant_column_mapping.csv
```

ตอบว่า:

1. `DATA` sheet เป็น matrix หรือ tidy table?
2. date อยู่ใน column ใด?
3. station IDs อยู่ใน rows หรือ column headers?
4. ทำไม PM₂.₅ จึงไม่มี source column เดียวชื่อ `PM25` ใน matrix ต้นฉบับ?
5. ถ้าเอา Excel หลายปี `pd.concat()` ก่อน `melt()` จะเกิดปัญหาอะไร?

---

## Exercise 3 — Missing ≠ Zero

เลือก station 1 แห่ง

นับ:

```text
PM2.5 missing days
PM2.5 valid days
```

อธิบายว่าทำไมจึงไม่ควรแทน missing ด้วย 0 โดยอัตโนมัติ

---

## Exercise 4 — Extreme PM₂.₅

ค้น 10 วัน PM₂.₅ สูงที่สุด

รายงาน:

```text
date
station
province
PM2.5
```

จากนั้นตอบ:

> เราสามารถสรุปได้หรือไม่ว่า 10 ค่านี้เป็น error?

คำตอบควรอธิบายว่าต้องตรวจข้อมูลอะไรเพิ่มเติม

---

## Exercise 5 — Mean vs Median

เลือก station 1 แห่ง แล้วคำนวณ:

```text
mean PM2.5
median PM2.5
```

ถ้าค่าต่างกันมาก ให้อธิบายว่า distribution อาจมีลักษณะอย่างไร

---

## Exercise 6 — Completeness

เลือก:

```text
สถานี coverage สูง 1 แห่ง
สถานี coverage ต่ำ 1 แห่ง
```

เปรียบเทียบค่าเฉลี่ย PM₂.₅

จากนั้นอภิปรายว่า:

> เราควรเปรียบเทียบสองสถานีนี้โดยดู mean อย่างเดียวหรือไม่?

---

## Exercise 7 — Three Provinces

เปรียบเทียบ:

```text
Saraburi
Lop Buri
Nakhon Nayok
```

ในเบื้องต้นด้วย table เท่านั้น

รายงาน:

```text
number of stations
station-days
mean PM2.5
median PM2.5
```

ยังไม่สรุปเชิง spatial จนกว่าเราจะเรียน GIS ในบทถัดไป

# Scientific Interpretation Checklist

ก่อนเขียนประโยค เช่น:

> “ปี 2024 มี PM₂.₅ สูงกว่าปี 2023”

ควรถามก่อน:

```text
จำนวนสถานีเท่ากันหรือไม่?
data coverage ใกล้เคียงกันหรือไม่?
ฤดูกาลที่มีข้อมูลเหมือนกันหรือไม่?
missingness ต่างกันหรือไม่?
station network เปลี่ยนหรือไม่?
aggregation เหมือนกันหรือไม่?
```

ดังนั้น descriptive statistics ไม่ควรถูกแยกออกจาก data-quality assessment

# Notebook ต่อไป

## Notebook 03 — Map Literacy, CRS and GeoPandas

ชื่อ:

```text
03_map_literacy_CRS_and_geopandas.ipynb
```

เราจะนำสิ่งที่เรียนใน Notebook 02:

```text
station_id
PM2.5
latitude
longitude
province
```

เข้าสู่แนวคิด GIS:

```text
Attribute
    +
Geometry
    ↓
GeoDataFrame
```

แล้วเรียน:

```text
Point
Polygon
Latitude / Longitude
EPSG:4326
CRS
Map extent
Province / Amphoe / Tambon
PCD station map
OGIMET station map
```

จากนั้นจึงเข้าสู่ spatial join, distance และ station matching ใน Notebook 04